Going to try with matplot lib

I used chatgpt to help me generate some of this initial code

In [2]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import matplotlib.colors as mcolors

Read in dataframes, and then make a list of dataframes so I can run through them all

In [3]:
df_2005 = pd.read_csv('DC-migration-2005.csv')
df_2009 = pd.read_csv('DC-migration-2009.csv')
df_2013 = pd.read_csv('DC-migration-2013.csv')
df_2017 = pd.read_csv('DC-migration-2017.csv')
df_2021 = pd.read_csv('DC-migration-2021.csv')



Drop the initial columns that were carried over from the indexes and messing up this data.

In [4]:
df_2005 = df_2005.drop(columns=["Unnamed: 0"])
df_2009 = df_2009.drop(columns=["Unnamed: 0"])
df_2013 = df_2013.drop(columns=["Unnamed: 0"])
df_2017 = df_2017.drop(columns=["Unnamed: 0"])
df_2021 = df_2021.drop(columns=["Unnamed: 0"])

In [5]:
dfs = [df_2005, df_2009, df_2013, df_2017, df_2021]

In [9]:
df_2005.tail()

,state,pop_migrated,MOE +/-,state_population,migrated_adjusted,moe_adjusted,winning_party,winner_percent_of_votes
44,Vermont,27,46,643077,0.000042,0.000072,DEMOCRAT,0.528211
45,Washington,443,354,7705281,0.000057,0.000046,DEMOCRAT,0.496997
46,West Virginia,389,376,1793716,0.000217,0.000210,REPUBLICAN,0.687396
47,Wisconsin,49,82,5893718,0.000008,0.000014,NaN,NaN
48,Wyoming,0,286,576851,0.000000,0.000496,NaN,NaN


Some code to help me adjust the colors based on party, and strength of color to represent how red or blue the state 

In [7]:
# this converts into RGB numbers between 0 and 

def adjust_color(color, factor):
    rgb = np.array(mcolors.to_rgb(color))
    return tuple(np.clip(rgb * factor, 0, 1))

BASE_COLORS = {
    "REPUBLICAN": "#d62728",
    "DEMOCRAT": "#1f77b4"
}

Make sure all the values I need are numeric.

In [ ]:
for df in dfs:
    df["migrated_adjusted"] = pd.to_numeric(df["migrated_adjusted"])
    df["moe_adjusted"] = pd.to_numeric(df["moe_adjusted"])
    df["winner_percent_of_votes"] = pd.to_numeric(df["winner_percent_of_votes"])



             state  pop_migrated  MOE +/-  state_population  \
0          Alabama             0      286           5024279   
1           Alaska             0      286            733391   
2          Arizona           490      573           7151502   
3         Arkansas             0      286           3011524   
4       California          2348     1010          39538223   
5         Colorado           269      250           5773714   
6      Connecticut           261      209           3605944   
7         Delaware           357      346            989948   
8          Florida          1103      461          21538187   
9          Georgia           319      209          10711908   
10          Hawaii            56       92           1455271   
11           Idaho             0      286           1839106   
12        Illinois          1519      812          12812508   
13         Indiana           192      141           6785528   
14            Iowa             0      286           319

KeyError: 'moe_adjusted'

Get the top 10 states with the most people who migrated into DC in that year

In [ ]:
for df in dfs:
    top10 = df.nlargest(10, "migrated_adjusted").sort_values("migrated_adjusted")

    plt.figure(figsize=(10, 6))

    plt.barh(
        top10["states"],
        top10["migrated_adjusted"],
        xerr=top10["moe_adjusted"],
        color=colors,
        alpha=0.9,
        capsize=4
    )


In [ ]:

    # color scaling (0.45 → light, 0.80 → dark)
    min_p = 0.45
    max_p = 0.80

    colors = []
    for _, row in top10.iterrows():
        base = BASE_COLORS.get(row["winning_party"], "#999999")

        # normalize percent into 0–1
        p = row["winner_percent_of_votes"]
        norm = (p - min_p) / (max_p - min_p)
        norm = np.clip(norm, 0, 1)

        # map to light/dark range
        factor = 0.5 + norm * 0.7  # tweakable intensity range

        colors.append(adjust_color(base, factor))

    # plot
    plt.figure(figsize=(10, 6))

    plt.barh(
        top10["states"],
        top10["migrated_adjusted"],
        xerr=top10["moe_adjusted"],
        color=colors,
        alpha=0.9,
        capsize=4
    )

    plt.title(f"Top 10 States Migrating to DC - {year}")
    plt.xlabel("Adjusted Migration to DC")
    plt.ylabel("State")

    plt.tight_layout()
    plt.show()



In [ ]:

years_files = {
    2005: "DC-migration-2005.csv",
    2009: "DC-migration-2009.csv",
    2013: "DC-migration-2013.csv",
    2017: "DC-migration-2017.csv",
    2021: "DC-migration-2021.csv"
}

for year, file in years_files.items():
    plot_year(file, year)